In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from pymcts.games.bridgit.game import BridgitGame
from pymcts.games.bridgit.player import Player
from pymcts.games.bridgit.visualizer import Visualizer
from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.games.bridgit.config import BoardConfig, NeuralNetConfig
from pymcts.core.mcts import MCTS
from pymcts.core.config import MCTSConfig
from plotly.subplots import make_subplots

In [ ]:
board_config = BoardConfig(size=4)
mcts_config = MCTSConfig(num_simulations=200000, solve_terminal=True)
net_config = NeuralNetConfig(num_channels=32, num_res_blocks=2)

net = BridgitNet(board_config, net_config)
mcts = MCTS(net, mcts_config)

game = BridgitGame(board_config)

g = board_config.grid_size
print(f"Board: {g}x{g}")
print(f"Legal moves: {len(game.valid_actions())}")
print(f"Device: {net.device}")

In [ ]:
game = BridgitGame(board_config)
game.make_action(game.row_col_to_action(1, 1))
game.make_action(game.row_col_to_action(2, 2))
game.make_action(game.row_col_to_action(3, 3))

# Get raw net prediction (1D policy)
policy, value = mcts._predict(game)

# Mask to only legal moves
mask = game.to_mask().float()
masked_policy = policy * mask
masked_policy /= masked_policy.sum()

# Reshape to 2D for visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["Raw Policy (all cells)", "Masked Policy (legal moves only)"])

fig.add_trace(Visualizer.visualize_array(policy.reshape(g, g), colorscale="Blues").data[0], row=1, col=1)
fig.add_trace(Visualizer.visualize_array(masked_policy.reshape(g, g), colorscale="Reds").data[0], row=1, col=2)

fig.update_layout(width=700, height=350, title=f"Net value estimate: {value:.3f}")
fig.show()

In [ ]:
game = BridgitGame(board_config)
game.make_action(game.row_col_to_action(1, 1))
game.make_action(game.row_col_to_action(2, 2))
game.make_action(game.row_col_to_action(3, 3))

# MCTS search
visit_counts = mcts._search(game).visit_counts(game.action_space_size).reshape(g, g)
mcts_probs = mcts.get_action_probs(game, temperature=1.0).reshape(g, g)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Visit Counts", "MCTS Policy (temp=1)"])

fig.add_trace(Visualizer.visualize_array(visit_counts, colorscale="Greens").data[0], row=1, col=1)
fig.add_trace(Visualizer.visualize_array(mcts_probs, colorscale="Reds").data[0], row=1, col=2)

fig.update_layout(width=700, height=350, title="MCTS concentrates visits on promising moves")
fig.show()

In [ ]:
mcts_probs

In [ ]:
board_config = BoardConfig(size=4)
game = BridgitGame(board_config)
# game.make_action(game.row_col_to_action(1, 1))
# game.make_action(game.row_col_to_action(1, 3))
# game.make_action(game.row_col_to_action(3, 3))
# game.make_action(game.row_col_to_action(1, 5))
# game.make_action(game.row_col_to_action(5, 3))
# game.make_action(game.row_col_to_action(1, 7))
Visualizer.visualize_game_state(game.get_state())

In [ ]:
root = mcts._search(game, verbose=True)

In [ ]:
for i in range(1000): 
    mcts.continue_search(root, 200000, verbose=True)

In [ ]:
root.q_value

In [ ]:
player_names = {0: "HORIZONTAL", 1: "VERTICAL"}

def find_game_overs(node, depth=0):
    if node.game.is_over:
        r, c = game.action_to_row_col(node.action) if node.action is not None else (None, None)
        action = f"({r},{c})" if node.action is not None else "root"
        winner = player_names.get(node.game.winner, "draw")
        print(f"{'  ' * depth}{action} depth={depth} winner={winner}")
    for child in node.children.values():
        find_game_overs(child, depth + 1)

find_game_overs(root)

In [ ]:
{move: (child.solved_value, child.visit_count, child.q_value) for move, child in root.children.items()}

In [ ]:
print(f"Root visits: {root.visit_count}")

In [ ]:
def max_depth(node, d=0):
    if not node.children:
        return d
    return max(max_depth(c, d+1) for c in node.children.values())

print(f"Max depth: {max_depth(root)}")

In [ ]:
def count_expanded(node):
    count = 1 if node.is_expanded else 0
    for child in node.children.values():
        count += count_expanded(child)
    return count

def count_terminal_visits(node):
    count = node.visit_count if node.game.is_over else 0
    for child in node.children.values():
        count += count_terminal_visits(child)
    return count

exp = count_expanded(root)
term = count_terminal_visits(root)
print(f"Expanded: {exp}, Terminal visits: {term}, Sum: {exp + term}, Root visits: {root.visit_count}")

In [ ]:
def find_deepest(node, d=0):
    if not node.children:
        return node, d
    return max((find_deepest(c, d+1) for c in node.children.values()), key=lambda x: x[1])

leaf, depth = find_deepest(root)
print(f"Depth: {depth}")
print(f"Game over: {leaf.game.is_over}")
print(f"Available moves: {len(leaf.game.valid_actions())}")
print(f"Is expanded: {leaf.is_expanded}")

In [ ]:
def count_solved(node, depth=0):
    """Print solved nodes in the tree."""
    if node.solved_value is not None:
        player = player_names.get(node.game.current_player, "?")
        r, c = game.action_to_row_col(node.action) if node.action is not None else (None, None)
        action = f"({r},{c})" if node.action is not None else "root"
        terminal = node.game.is_over
        reason = "terminal" if terminal else "inherited"
        print(f"{'  ' * depth}{action} depth={depth} solved={node.solved_value:+.0f} player={player} ({reason})")
    for child in node.children.values():
        count_solved(child, depth + 1)

count_solved(root)

In [ ]:
def find_at_depth(node, target, d=0):
    if d == target:
        return [node]
    result = []
    for c in node.children.values():
        result.extend(find_at_depth(c, target, d+1))
    return result

depth_11 = find_at_depth(root, 11)
print(f"Nodes at depth 11: {len(depth_11)}")
print(f"Solved: {sum(1 for n in depth_11 if n.solved_value is not None)}")
print(f"Expanded: {sum(1 for n in depth_11 if n.is_expanded)}")
for n in depth_11[:5]:
    terminal_children = sum(1 for c in n.children.values() if c.game.is_over)
    print(f"  solved={n.solved_value} expanded={n.is_expanded} children={len(n.children)} terminal_children={terminal_children}")

In [ ]:
for d in range(7,11):
    nodes = find_at_depth(root, d)
    expanded = sum(1 for n in nodes if n.is_expanded)
    solved = sum(1 for n in nodes if n.solved_value is not None)
    game_overs = sum(1 for n in nodes if n.game.is_over)
    print(f"Depth {d}: {len(nodes)} nodes, {expanded} expanded, {solved} solved, {game_overs} game_over")

In [ ]:
print(f"Root solved: {root.solved_value}")
print(f"Root visits: {root.visit_count}")

# How many of root's children are solved?
for action, child in root.children.items():
    r, c = game.action_to_row_col(action)
    print(f"  ({r},{c}) visits={child.visit_count} q={child.q_value:.3f} solved={child.solved_value}")

In [ ]:
visit_counts = root.visit_counts(game.action_space_size).reshape(g, g)
Visualizer.visualize_array(visit_counts, colorscale="Greens")

In [ ]:
probs = MCTS.visit_counts_to_probs(root.visit_counts(game.action_space_size), 0)

In [ ]:
action = torch.multinomial(probs.flatten(), 1).item()
row, col = game.action_to_row_col(action)

In [ ]:
print(f"Selected action {action} -> row={row}, col={col}")